# Chapter 11: Inferential Statistics & Hypothesis Testing

**ISM4641 - Python for Business Analytics**
**University of South Florida**
**Spring 2026**

---

## Introduction

You've all taken a statistics course before. This chapter is a **refresher** — but with a twist. Instead of memorizing formulas, we're going to build **visual intuition** for *why* statistical inference works.

The central idea: **the Central Limit Theorem** tells us that sample averages follow a predictable pattern (a bell curve), even when the underlying data doesn't. This one fact is the foundation for nearly all hypothesis testing.

By the end of this chapter, you'll understand:
- Why sample means form a normal distribution (and why that matters)
- What p-values and alpha levels *really* represent — visually
- How to run and interpret the three main types of t-tests in Python
- How to apply these ideas to real business decisions like A/B testing

## Table of Contents

- [11.1 Quick Descriptive Stats Refresher](#111-quick-descriptive-stats-refresher)
- [11.2 The Big Question: From Sample to Population](#112-the-big-question-from-sample-to-population)
- [11.3 The Central Limit Theorem — Visually](#113-the-central-limit-theorem--visually)
- [11.4 How Sample Size Changes Everything](#114-how-sample-size-changes-everything)
- [11.5 From Distributions to Decisions](#115-from-distributions-to-decisions)
- [11.6 Z-Scores and T-Values — The Test Statistics](#116-z-scores-and-t-values--the-test-statistics)
- [11.7 Alpha, P-Values, and the Decision Framework](#117-alpha-p-values-and-the-decision-framework)
- [11.8 Hypothesis Testing Framework](#118-hypothesis-testing-framework)
- [11.9 One-Sample t-Test](#119-one-sample-t-test)
- [11.10 Two-Sample t-Test](#1110-two-sample-t-test)
- [11.11 Paired t-Test](#1111-paired-t-test)
- [11.12 A/B Testing in Business](#1112-ab-testing-in-business)
- [11.13 Common Pitfalls and Best Practices](#1113-common-pitfalls-and-best-practices)
- [11.14 Summary](#1114-summary)
- [Looking Ahead: Week 12](#looking-ahead-week-12)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

# Display settings
pd.set_option('display.precision', 2)
%matplotlib inline

# Data repository URL for course datasets
DATA_URL = "https://raw.githubusercontent.com/prof-tcsmith/4641s26-data/main"

print("Libraries imported successfully!")

---

## 11.1 Quick Descriptive Stats Refresher

Before we get to inference, let's make sure we're on the same page with the basics. Descriptive statistics **summarize what you have**. They answer: *"What does my data look like?"*

| Measure | What It Tells You | Python |
|---------|-------------------|--------|
| **Mean** | The average — sensitive to outliers | `df['col'].mean()` |
| **Median** | The middle value — robust to outliers | `df['col'].median()` |
| **Std Dev** | How spread out values are | `df['col'].std()` |
| **Correlation** | Strength of linear relationship between two variables | `df['col1'].corr(df['col2'])` |
| **describe()** | All-in-one summary | `df.describe()` |

The key thing to remember: descriptive stats tell you about your **sample**. They say nothing about the broader **population** — that's what inferential statistics is for.

In [ ]:
# Quick descriptive stats demo
np.random.seed(42)

# Simulated business data: customer spending
spending = pd.Series(np.random.normal(75, 20, 200), name='Spending')

print("Quick Summary:")
print(f"  Mean:    ${spending.mean():.2f}")
print(f"  Median:  ${spending.median():.2f}")
print(f"  Std Dev: ${spending.std():.2f}")
print(f"  Min:     ${spending.min():.2f}")
print(f"  Max:     ${spending.max():.2f}")
print()
print("Full summary via .describe():")
print(spending.describe())

In [ ]:
# Visualize with mean and median marked
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: normal-ish data — mean ≈ median
axes[0].hist(spending, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].axvline(spending.mean(), color='red', linewidth=2, linestyle='--',
                label=f'Mean: ${spending.mean():.0f}')
axes[0].axvline(spending.median(), color='green', linewidth=2, linestyle=':',
                label=f'Median: ${spending.median():.0f}')
axes[0].set_title('Symmetric Data: Mean ≈ Median', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Customer Spending ($)')
axes[0].legend(fontsize=10)

# Right: skewed data — mean ≠ median
np.random.seed(42)
skewed = pd.Series(np.random.exponential(50, 200), name='Income')
axes[1].hist(skewed, bins=25, color='coral', edgecolor='white', alpha=0.7)
axes[1].axvline(skewed.mean(), color='red', linewidth=2, linestyle='--',
                label=f'Mean: ${skewed.mean():.0f}')
axes[1].axvline(skewed.median(), color='green', linewidth=2, linestyle=':',
                label=f'Median: ${skewed.median():.0f}')
axes[1].set_title('Skewed Data: Mean Pulled by Outliers', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()
print("\nKey takeaway: Use median when data is skewed. Use mean when data is symmetric.")

### Correlation Quick Recap

Correlation measures the **strength and direction** of a linear relationship between two variables.

| Value | Interpretation |
|-------|----------------|
| +1.0 | Perfect positive — as X goes up, Y goes up |
| 0.0 | No linear relationship |
| -1.0 | Perfect negative — as X goes up, Y goes down |

**Critical reminder:** Correlation does NOT imply causation. Ice cream sales and drowning deaths are correlated — but ice cream doesn't cause drowning. Both increase in summer.

Descriptive stats (including correlation) summarize your data. To make **claims about the population**, we need inferential statistics — and that starts with understanding the Central Limit Theorem.

---

## 11.2 The Big Question: From Sample to Population

**We almost never get to see the full population.** We work with samples — subsets of the data we actually care about.

- A survey of 500 customers (out of 2 million)
- Sales data from 30 stores (out of 300)
- Response times from 100 support tickets (out of thousands)

**The fundamental question of inferential statistics:**

> *Can we make reliable claims about the population from just a sample?*

**Yes — and the reason why is one of the most powerful results in all of statistics.** Let's build up to it visually.

---

## 11.3 The Central Limit Theorem — Visually

Let's imagine we *can* see an entire population. We'll use this to understand what happens when we take samples from it.

In [ ]:
# Create a population that is clearly NOT normal
# Think of this as: wait times, claim amounts, incomes, etc.
np.random.seed(42)
population = np.random.exponential(scale=50, size=100000)

plt.figure(figsize=(12, 5))
plt.hist(population, bins=80, color='steelblue', edgecolor='white', alpha=0.7)
plt.axvline(np.mean(population), color='red', linewidth=2, linestyle='--',
            label=f'Population Mean: {np.mean(population):.1f}')
plt.title('The Full Population — Clearly Not a Bell Curve',
          fontsize=14, fontweight='bold')
plt.xlabel('Value (e.g., wait time in seconds)')
plt.ylabel('Frequency')
plt.legend(fontsize=12)
plt.show()

print(f"Population size: {len(population):,}")
print(f"Population mean: {np.mean(population):.2f}")
print(f"Population std dev: {np.std(population):.2f}")
print(f"\nThis distribution is right-skewed — most values are small,")
print(f"but there's a long tail of large values. Definitely not normal!")

### Taking Random Samples

Now let's take a few random samples of 30 values each and compute their means. Notice how each sample gives us a *different* mean — but they all cluster around the true population mean.

In [ ]:
# Take 6 random samples and look at each one
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Six Random Samples (n=30) from the Same Skewed Population',
             fontsize=14, fontweight='bold')

sample_means = []
for i, ax in enumerate(axes.flat):
    sample = np.random.choice(population, size=30)
    sample_means.append(np.mean(sample))
    ax.hist(sample, bins=15, color='steelblue', edgecolor='white', alpha=0.7)
    ax.axvline(np.mean(sample), color='red', linewidth=2, linestyle='--')
    ax.set_title(f'Sample {i+1}: Mean = {np.mean(sample):.1f}', fontsize=11)
    ax.set_xlim(0, 250)

plt.tight_layout()
plt.show()

print(f"\nSample means: {[f'{m:.1f}' for m in sample_means]}")
print(f"True population mean: {np.mean(population):.1f}")
print(f"\nEach sample looks different. Each mean is different.")
print(f"But they're all CLUSTERING around the true value!")

### The Magic: Plotting Thousands of Sample Means

What if we repeated this experiment 3,000 times? Take a sample of 30, compute the mean, record it. Do this 3,000 times and plot all the means.

**Watch what shape emerges:**

In [ ]:
# Take 3,000 samples of size 30 and collect all the means
np.random.seed(42)
sample_means_3000 = [np.mean(np.random.choice(population, size=30))
                     for _ in range(3000)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: the original population (NOT normal)
axes[0].hist(population, bins=80, color='steelblue', edgecolor='white',
             alpha=0.7, density=True)
axes[0].set_title('The Population\n(Definitely Not Normal)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')

# Right: distribution of sample means (NORMAL!)
axes[1].hist(sample_means_3000, bins=50, color='coral', edgecolor='white',
             alpha=0.7, density=True)
mu = np.mean(sample_means_3000)
sigma = np.std(sample_means_3000)
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
axes[1].plot(x, norm.pdf(x, mu, sigma), 'k-', linewidth=2,
             label='Normal curve overlay')
axes[1].set_title('Distribution of 3,000 Sample Means\n(It\'s Normal!)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sample Mean')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("The population is skewed (left plot).")
print("But the distribution of SAMPLE MEANS is a perfect bell curve (right plot)!")
print(f"\nMean of sample means: {mu:.2f} (true population mean: {np.mean(population):.2f})")
print(f"The sample means center on the true population mean!")

### This is the Central Limit Theorem (CLT)

**No matter what the population looks like — skewed, bimodal, uniform, anything — the distribution of sample means will be approximately normal**, as long as the sample size is large enough.

This is the **Central Limit Theorem**, and it's *the reason statistics works*.

| What CLT says | Why it matters |
|---------------|----------------|
| Sample means form a bell curve | We can use the normal distribution to make predictions |
| The center is the true population mean | Sample means cluster around the right answer |
| Larger samples → tighter bell curve | Bigger samples give more precise estimates |

**You don't need a normal population to do normal-based testing.** The CLT gives you that for free.

### CLT Works for ANY Population Shape

Let's prove this with three very different population shapes: skewed, bimodal, and uniform.

In [ ]:
# CLT works regardless of population shape
np.random.seed(42)

populations = {
    'Right-Skewed\n(Exponential)': np.random.exponential(50, 100000),
    'Bimodal\n(Two Peaks)': np.concatenate([
        np.random.normal(30, 5, 50000),
        np.random.normal(70, 5, 50000)
    ]),
    'Uniform\n(Completely Flat)': np.random.uniform(0, 100, 100000)
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('CLT Works for ANY Population Shape (n=30, 2000 samples each)',
             fontsize=14, fontweight='bold')

for j, (name, pop) in enumerate(populations.items()):
    # Top row: the population itself
    axes[0, j].hist(pop, bins=60, color='steelblue', edgecolor='white',
                    alpha=0.7, density=True)
    axes[0, j].set_title(f'Population: {name}', fontsize=11, fontweight='bold')

    # Bottom row: distribution of sample means
    means = [np.mean(np.random.choice(pop, size=30)) for _ in range(2000)]
    axes[1, j].hist(means, bins=40, color='coral', edgecolor='white',
                    alpha=0.7, density=True)
    mu, sigma = np.mean(means), np.std(means)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    axes[1, j].plot(x, norm.pdf(x, mu, sigma), 'k-', linewidth=2)
    axes[1, j].set_title('Sample Means → Normal!', fontsize=11, fontweight='bold')

axes[0, 0].set_ylabel('Population', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Sample Means', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top row: Three very different population shapes.")
print("Bottom row: The distribution of sample means is ALWAYS normal!")
print("\nThis is the power of the CLT — it works regardless of the original distribution.")

---

## 11.4 How Sample Size Changes Everything

The CLT tells us two more important things:
1. **Larger samples** → the bell curve of means gets **narrower** (more precise)
2. **More repetitions** → the histogram looks **smoother** (higher resolution)

Let's see both effects visually.

In [ ]:
# Effect of sample size on the spread of sample means
np.random.seed(42)
sample_sizes = [5, 15, 30, 100]
n_samples = 3000

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle('Larger Sample Size → Tighter Distribution of Means',
             fontsize=14, fontweight='bold')

for ax, n in zip(axes, sample_sizes):
    means = [np.mean(np.random.choice(population, size=n))
             for _ in range(n_samples)]
    ax.hist(means, bins=45, color='coral', edgecolor='white',
            alpha=0.7, density=True)
    mu, sigma = np.mean(means), np.std(means)
    x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ax.plot(x_range, norm.pdf(x_range, mu, sigma), 'k-', linewidth=2)
    ax.set_title(f'n = {n}\nSpread: ±{sigma:.1f}', fontsize=11)
    ax.set_xlabel('Sample Mean')
    ax.axvline(np.mean(population), color='red', linestyle='--', alpha=0.5)
    ax.set_xlim(20, 80)

plt.tight_layout()
plt.show()

print("Red dashed line = true population mean")
print("\nAs sample size increases:")
print("  n=5:   Means are spread widely — lots of uncertainty")
print("  n=15:  Getting tighter")
print("  n=30:  Much more precise")
print("  n=100: Very tightly clustered around the true mean")

In [ ]:
# Effect of number of samples on resolution
np.random.seed(42)
num_samples_list = [30, 200, 1000, 5000]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle('More Samples = Higher Resolution (each sample has n=30)',
             fontsize=14, fontweight='bold')

for ax, n_samp in zip(axes, num_samples_list):
    means = [np.mean(np.random.choice(population, size=30))
             for _ in range(n_samp)]
    ax.hist(means, bins=30, color='coral', edgecolor='white',
            alpha=0.7, density=True)
    mu, sigma = np.mean(means), np.std(means)
    x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ax.plot(x_range, norm.pdf(x_range, mu, sigma), 'k-', linewidth=2)
    ax.set_title(f'{n_samp} Samples', fontsize=11)
    ax.set_xlabel('Sample Mean')
    ax.set_xlim(20, 80)

plt.tight_layout()
plt.show()

print("Same sample size (n=30), but different number of repetitions:")
print("  30 samples:   Choppy — hard to see the shape")
print("  200 samples:  Starting to look normal")
print("  1000 samples: Clearly bell-shaped")
print("  5000 samples: Smooth and precise")
print("\nThe underlying shape is ALWAYS normal — more samples just reveal it more clearly.")

### Don't Confuse These Two Ideas

| Concept | What It Controls | Analogy |
|---------|------------------|---------|
| **Sample size (n)** | **Width** of the bell curve — how precise your estimate is | "How much data did I collect?" |
| **Number of samples** | **Resolution** — how smooth the histogram looks | "How many times did I repeat the experiment?" |

**In practice, you usually take just one sample.** But the CLT tells you *where your sample mean would fall* on the bell curve of all possible sample means — and that's enough to make decisions.

### One Sample Is All You Get

In practice, **you don't repeat the study 3,000 times.** You collect *one* sample and get *one* mean. But the CLT tells you what the full distribution of means *would* look like — so you can judge how unusual your single result is.

Each "study" below takes just one sample of 30 values and plots where that sample mean falls on the theoretical bell curve of all possible sample means. You never actually see the gray curve — but the CLT *guarantees* it's there. That's why a single sample is enough to make a decision.

In [ ]:
# One Sample Is All You Get — three different "studies," each with one sample
np.random.seed(42)

# The CLT-based sampling distribution (what we KNOW from theory)
pop_mean = np.mean(population)
se = np.std(population) / np.sqrt(30)  # standard error for n=30
x_curve = np.linspace(pop_mean - 4*se, pop_mean + 4*se, 300)
y_curve = norm.pdf(x_curve, pop_mean, se)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
fig.suptitle('Three Different Studies — Each Takes Just One Sample (n=30)',
             fontsize=14, fontweight='bold')

seeds = [7, 21, 88]
labels = ['Study A', 'Study B', 'Study C']
colors = ['#2471a3', '#e67e22', '#27ae60']

for ax, seed, label, color in zip(axes, seeds, labels, colors):
    np.random.seed(seed)
    sample = np.random.choice(population, size=30)
    sample_mean = np.mean(sample)

    # Show the theoretical sampling distribution (the "invisible" bell curve)
    ax.plot(x_curve, y_curve, 'k-', linewidth=2, alpha=0.4,
            label='All possible\nsample means')
    ax.fill_between(x_curve, y_curve, alpha=0.08, color='gray')

    # Mark where THIS sample's mean lands
    ax.axvline(sample_mean, color=color, linewidth=3,
               label=f'{label}: x\u0304 = {sample_mean:.1f}')
    ax.plot(sample_mean, norm.pdf(sample_mean, pop_mean, se),
            'o', color=color, markersize=12, zorder=5)

    ax.axvline(pop_mean, color='red', linestyle='--', alpha=0.4,
               label=f'True \u03bc = {pop_mean:.1f}')
    ax.set_title(f'{label}\nx\u0304 = {sample_mean:.1f}', fontsize=12,
                 fontweight='bold', color=color)
    ax.set_xlabel('Sample Mean')
    ax.set_xlim(x_curve[0], x_curve[-1])
    ax.legend(fontsize=8, loc='upper right')

axes[0].set_ylabel('Probability Density')
plt.tight_layout()
plt.show()

print("Each study gets ONE dot on the bell curve.")
print("You never see the gray curve directly — but the CLT guarantees it's there.")
print("That's why a single sample is enough to make a decision.")

---

## 11.5 From Distributions to Decisions

Here's the key insight that connects the CLT to hypothesis testing:

> If we know that sample means follow a normal distribution centered on the true population mean, then we can ask: **"If the true mean were X, how likely is it that I'd get a sample mean this far away?"**

If our observed sample mean falls in the *middle* of the bell curve → it's consistent with the hypothesis.
If it falls way out in the *tails* → something doesn't add up.

Let's visualize this.

In [ ]:
# Visualizing: "Where does our sample mean fall?"
x = np.linspace(35, 65, 500)
y = norm.pdf(x, 50, 5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: sample mean close to hypothesized value — plausible
axes[0].plot(x, y, 'k-', linewidth=2)
axes[0].fill_between(x, y, alpha=0.12, color='steelblue')
axes[0].axvline(50, color='gray', linestyle='--', alpha=0.5)
observed1 = 52
axes[0].axvline(observed1, color='green', linewidth=2.5,
                label=f'Sample Mean = {observed1}')
axes[0].annotate('This is plausible\nunder the hypothesis',
                 xy=(observed1, 0.06), xytext=(56, 0.065),
                 arrowprops=dict(arrowstyle='->', color='green', lw=1.5),
                 fontsize=11, color='green', fontweight='bold')
axes[0].set_title('Sample Mean Near Center\n→ Consistent with H₀',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Possible Sample Means')
axes[0].legend(fontsize=10)

# Right: sample mean far from hypothesized value — suspicious
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, alpha=0.12, color='steelblue')
axes[1].axvline(50, color='gray', linestyle='--', alpha=0.5)
observed2 = 62
axes[1].axvline(observed2, color='red', linewidth=2.5,
                label=f'Sample Mean = {observed2}')
axes[1].annotate('This seems very unlikely\nif the true mean is 50!',
                 xy=(observed2, 0.002), xytext=(55, 0.05),
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                 fontsize=11, color='red', fontweight='bold')
axes[1].set_title('Sample Mean in the Tail\n→ Evidence Against H₀',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Possible Sample Means')
axes[1].legend(fontsize=10)

for ax in axes:
    ax.set_ylabel('Probability Density')

plt.tight_layout()
plt.show()

print("If our sample mean falls near the center → the hypothesis is plausible.")
print("If it falls far out in the tails → the hypothesis is suspect.")
print("\nThis is the core logic of hypothesis testing!")

---

## 11.6 Z-Scores and T-Values — The Test Statistics

We've seen that if our sample mean falls far from the hypothesized value, that's evidence against H₀. But **"far" is relative** — a gap of 9 minutes means something very different for delivery times vs. heart surgery times. We need a universal yardstick.

### What is a Z-Score?

The **z-score** standardizes the distance between your sample mean and the hypothesized value into **"how many standard deviations away?"**

$$z = \frac{\bar{x} - \mu_0}{\sigma \, / \, \sqrt{n}}$$

Where:
- **x̄** = your sample mean (what you observed)
- **μ₀** = the hypothesized population mean (what you're testing against)
- **σ** = the population standard deviation (how spread out the population is)
- **n** = your sample size
- **σ / √n** = the **standard error** — how much sample means naturally vary

**Interpretation:** |z| ≈ 0 means your result is unsurprising. |z| > 2 means it's unusual (~5% territory). |z| > 3 means it's very unusual — strong evidence against H₀.

### Z-Score in Action

**Example:** A delivery company claims their average delivery time is **50 minutes**. You sample **36 deliveries** and get a sample mean of **x̄ = 59 minutes**. The population standard deviation is known to be **σ = 18 minutes**.

$$z = \frac{59 - 50}{18 \, / \, \sqrt{36}} = \frac{9}{3.0} = 3.0$$

Our sample mean is **3 standard deviations** away from the claimed value. That's very far out in the tail — strong evidence that the real average is not 50 minutes.

In [ ]:
# Z-score worked example: delivery time claim
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Setup
mu_0 = 50       # hypothesized mean (the company's claim)
sigma = 18      # population std dev (known)
n = 36          # sample size
x_bar = 59      # our observed sample mean
se = sigma / np.sqrt(n)  # standard error = 18/6 = 3.0
z_obs = (x_bar - mu_0) / se  # z = (59-50)/3 = 3.0

# Left panel: raw scale — sampling distribution if H₀ is true
x_raw = np.linspace(mu_0 - 4*se, mu_0 + 4*se, 300)
y_raw = norm.pdf(x_raw, mu_0, se)

axes[0].plot(x_raw, y_raw, 'k-', linewidth=2)
axes[0].fill_between(x_raw, y_raw, alpha=0.1, color='steelblue')
axes[0].axvline(mu_0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x_bar, color='red', linewidth=2.5, label=f'Our x\u0304 = {x_bar}')
axes[0].set_title('Raw Scale\n(Sampling Distribution if H\u2080 True)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sample Mean (minutes)')
axes[0].legend(fontsize=11)

# Right panel: standardized z-score scale
x_z = np.linspace(-4, 4, 300)
y_z = norm.pdf(x_z)

axes[1].plot(x_z, y_z, 'k-', linewidth=2)
axes[1].fill_between(x_z, y_z, alpha=0.1, color='steelblue')
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(z_obs, color='red', linewidth=2.5,
                label=f'z = ({x_bar}\u2212{mu_0}) / ({sigma}/\u221a{n}) = {z_obs:.1f}')
axes[1].annotate(f'z = {z_obs:.1f}\n3 std devs away!',
                xy=(2.85, 0.009), xytext=(1.0, 0.25),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2),
                fontsize=10, color='#27ae60', fontweight='bold', zorder=10)
axes[1].set_title('Standardized Scale\n(Z-Score)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Standard Deviations from Expected (z)')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Standard error = \u03c3 / \u221an = {sigma} / \u221a{n} = {se:.1f}")
print(f"z = (x\u0304 - \u03bc\u2080) / SE = ({x_bar} - {mu_0}) / {se:.1f} = {z_obs:.1f}")
print(f"\nInterpretation: Our sample mean is {z_obs:.1f} standard deviations away")
print(f"from the claimed value. That's very unusual — strong evidence against H\u2080.")

### From Z to T — When σ Is Unknown

In practice, **we almost never know σ** (the true population standard deviation). Instead, we estimate it from the sample using **s** (the sample standard deviation). This introduces extra uncertainty:

$$t = \frac{\bar{x} - \mu_0}{s \, / \, \sqrt{n}}$$

The formula is identical — we just swap σ for s. But because **s** is an *estimate* that varies from sample to sample, the resulting distribution has **heavier tails** than the normal curve. This is the **t-distribution**, and it uses **degrees of freedom (df) = n − 1**.

**Why n − 1?** One degree of freedom is "used up" to estimate the mean, leaving n − 1 independent pieces of information for estimating variability.

As your sample size grows, s becomes a better estimate of σ, the t-distribution approaches the normal, and the z-score and t-value converge.

In [ ]:
# T-distribution vs. Normal: visual comparison
x = np.linspace(-5, 5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: overlay multiple t-distributions against normal
for n_size, color, ls in [(4, '#e74c3c', '-'), (11, '#e67e22', '--'),
                           (31, '#27ae60', '-.')]:
    df_val = n_size - 1
    axes[0].plot(x, stats.t.pdf(x, df_val), color=color, linewidth=2,
                linestyle=ls, label=f't (n={n_size}, df={df_val})')
axes[0].plot(x, norm.pdf(x), 'k-', linewidth=2.5, label='Normal (z)')
axes[0].set_title('T-Distribution Approaches Normal\nas Sample Size Grows',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Test Statistic Value')
axes[0].set_ylabel('Probability Density')
axes[0].legend(fontsize=10)

# Right: zoom in on tails to show the difference
axes[1].plot(x, norm.pdf(x), 'k-', linewidth=2.5, label='Normal (z)')
axes[1].plot(x, stats.t.pdf(x, 3), color='#e74c3c', linewidth=2,
            label='t (n=4, df=3)')
axes[1].fill_between(x, stats.t.pdf(x, 3), norm.pdf(x),
                     where=(np.abs(x) > 2), alpha=0.3, color='#e74c3c')
axes[1].set_title('The Tails Tell the Story\n(Shaded = Extra Uncertainty with Small n)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Test Statistic Value')
axes[1].set_xlim(-5, 5)
axes[1].legend(fontsize=10)
axes[1].annotate('Heavier tails →\nharder to reject H\u2080',
                xy=(3.2, 0.02), fontsize=11, color='#e74c3c',
                fontweight='bold')

plt.tight_layout()
plt.show()

print("With small samples (df=3), the t-distribution has fatter tails.")
print("This means you need a more extreme test statistic to reject H\u2080.")
print("\nAs df grows (df=10, df=30), the t-distribution looks more and more")
print("like the normal curve. By df=30+ they're nearly identical.")

### From Test Statistic to Decision

The test statistic is the bridge between your data and your conclusion. Here's the pipeline:

1. **Collect data** — compute x̄, s, and n from your sample
2. **Compute t** — using t = (x̄ − μ₀) / (s / √n)
3. **Find p-value** — the area in the tails beyond |t| on the t-distribution
4. **Decide** — if p < α, reject H₀; if p ≥ α, fail to reject H₀

**Key insight:** A larger |t| means your sample mean is further from the hypothesized value (in standard-error units) → further into the tails → smaller p-value → stronger evidence against H₀.

**You don't compute p-values by hand** — `scipy` does it for you. When you call `stats.ttest_1samp()`, `stats.ttest_ind()`, or `stats.ttest_rel()`, the function returns both the t-statistic *and* the p-value automatically.

---

## 11.7 Alpha, P-Values, and the Decision Framework

Now let's formalize the intuition from the previous section.

### What is Alpha (α)?

**Alpha is the threshold we set *before* looking at the data.** It defines how far out in the tails a result needs to be before we call it "statistically significant."

- Typically α = 0.05 (5%) — the most common standard
- This means: "I'll accept a 5% chance of falsely rejecting H₀"
- The shaded regions in the tails represent this threshold

In [ ]:
# Visualizing alpha = 0.05 (two-tailed)
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, y, 'k-', linewidth=2)

# Shade rejection regions
ax.fill_between(x, y, where=(x <= -1.96), alpha=0.4, color='red')
ax.fill_between(x, y, where=(x >= 1.96), alpha=0.4, color='red')
ax.fill_between(x, y, where=(x > -1.96) & (x < 1.96),
                alpha=0.1, color='steelblue')

ax.axvline(-1.96, color='red', linestyle='--', alpha=0.7)
ax.axvline(1.96, color='red', linestyle='--', alpha=0.7)

ax.annotate('Reject H₀\n(2.5%)', xy=(-2.8, 0.015), fontsize=13,
            color='red', fontweight='bold', ha='center')
ax.annotate('Reject H₀\n(2.5%)', xy=(2.8, 0.015), fontsize=13,
            color='red', fontweight='bold', ha='center')
ax.annotate('Fail to Reject H₀\n(95%)', xy=(0, 0.17), fontsize=14,
            color='steelblue', fontweight='bold', ha='center')

ax.set_title('α = 0.05 — The Decision Boundaries (Two-Tailed Test)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Standard Deviations from Expected Mean')
ax.set_ylabel('Probability Density')
plt.show()

print("The red shaded regions are the 'rejection zones.'")
print("If our test statistic falls in the red → reject H₀.")
print("If it falls in the blue → fail to reject H₀.")
print("\nα = 0.05 means 2.5% in each tail (for a two-tailed test).")

### Comparing Different Alpha Levels

A stricter alpha means you need stronger evidence to reject H₀. Think of it as setting the bar higher for what counts as "significant." 

In [ ]:
# Compare different alpha levels
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)
alphas = [0.10, 0.05, 0.01]
z_crits = [1.645, 1.96, 2.576]
colors = ['#f39c12', '#e74c3c', '#8e44ad']
labels = ['Lenient', 'Standard', 'Strict']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Different Alpha Levels = Different Standards of Evidence',
             fontsize=14, fontweight='bold')

for ax, alpha, z, color, label in zip(axes, alphas, z_crits, colors, labels):
    ax.plot(x, y, 'k-', linewidth=2)
    ax.fill_between(x, y, where=(x <= -z), alpha=0.4, color=color)
    ax.fill_between(x, y, where=(x >= z), alpha=0.4, color=color)
    ax.fill_between(x, y, where=(x > -z) & (x < z),
                    alpha=0.08, color='steelblue')
    ax.axvline(-z, color=color, linestyle='--', alpha=0.7)
    ax.axvline(z, color=color, linestyle='--', alpha=0.7)
    ax.set_title(f'α = {alpha} ({label})\nCritical z = ±{z}', fontsize=12)
    ax.set_xlabel('Standard Deviations')

plt.tight_layout()
plt.show()

print("α = 0.10: Easier to reject H₀ (more false positives, fewer false negatives)")
print("α = 0.05: The standard in most fields")
print("α = 0.01: Harder to reject H₀ (fewer false positives, more false negatives)")

### What is a P-Value?

The **p-value** answers a specific question:

> *"If nothing interesting were happening (H₀ is true), how likely would it be to get a result at least this extreme?"*

- **Small p-value** (e.g., 0.003) → Our result would be very unusual under H₀ → Strong evidence against H₀
- **Large p-value** (e.g., 0.42) → Our result is totally normal under H₀ → No evidence against H₀

**The p-value is the shaded area in the tails beyond our test statistic.**

In [ ]:
# Visualizing p-values: significant vs. not significant
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('P-Value = The Shaded Area Beyond the Test Statistic',
             fontsize=14, fontweight='bold')

# Left: small p-value (significant)
test_stat = 2.3
axes[0].plot(x, y, 'k-', linewidth=2)
axes[0].fill_between(x, y, where=(x >= test_stat), alpha=0.5, color='coral')
axes[0].fill_between(x, y, where=(x <= -test_stat), alpha=0.5, color='coral')
axes[0].axvline(test_stat, color='red', linewidth=2)
axes[0].axvline(-test_stat, color='red', linewidth=2)
p_val = 2 * (1 - norm.cdf(test_stat))
axes[0].set_title(f'Test Statistic = {test_stat}\np-value = {p_val:.4f} → Significant!',
                  fontsize=12, fontweight='bold')
axes[0].annotate('p-value\n(tiny shaded area)', xy=(3.0, 0.008), fontsize=11,
                 color='red', fontweight='bold')

# Right: large p-value (not significant)
test_stat2 = 0.8
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, where=(x >= test_stat2), alpha=0.5, color='coral')
axes[1].fill_between(x, y, where=(x <= -test_stat2), alpha=0.5, color='coral')
axes[1].axvline(test_stat2, color='red', linewidth=2)
axes[1].axvline(-test_stat2, color='red', linewidth=2)
p_val2 = 2 * (1 - norm.cdf(test_stat2))
axes[1].set_title(f'Test Statistic = {test_stat2}\np-value = {p_val2:.4f} → Not Significant',
                  fontsize=12, fontweight='bold')
axes[1].annotate('p-value\n(large shaded area)', xy=(1.6, 0.06), fontsize=11,
                 color='red', fontweight='bold')

for ax in axes:
    ax.set_xlabel('Standard Deviations from Expected Mean')
plt.tight_layout()
plt.show()

print(f"Left:  Test stat = {test_stat} → p-value = {p_val:.4f} → LESS than 0.05 → Significant")
print(f"Right: Test stat = {test_stat2} → p-value = {p_val2:.4f} → GREATER than 0.05 → Not significant")
print(f"\nThe p-value IS the shaded area. Smaller area = more extreme result = stronger evidence.")

### The Decision Rule

**Compare p-value to alpha:**

| If... | Then... | In Plain English |
|-------|---------|-----------------|
| p < α (0.05) | **Reject H₀** | "This result is too unlikely to be just chance" |
| p ≥ α (0.05) | **Fail to reject H₀** | "We don't have enough evidence to rule out chance" |

**Important nuances:**

- "Fail to reject" ≠ "H₀ is true" — we just lack sufficient evidence
- A small p-value tells you the result is *unlikely under H₀*, not that the effect is *large*
- α must be set *before* you look at the data — no peeking and adjusting!

### Type I and Type II Errors

| | H₀ is Actually True | H₀ is Actually False |
|---|---|---|
| **We Reject H₀** | Type I Error (False Positive) | Correct! |
| **We Fail to Reject H₀** | Correct! | Type II Error (False Negative) |

**Business examples:**
- **Type I Error:** Concluding a new campaign works when it doesn't → waste money on an ineffective campaign
- **Type II Error:** Concluding a campaign doesn't work when it does → miss a profitable opportunity

---

## 11.8 Hypothesis Testing Framework

Now let's put it all together into a formal procedure:

### The Five-Step Process

1. **State your hypotheses**
   - **H₀ (null):** "Nothing interesting is happening" — the default assumption
   - **H₁ (alternative):** "Something IS happening" — what you suspect

2. **Choose your significance level** (typically α = 0.05)

3. **Collect data and compute the test statistic**

4. **Find the p-value** — how extreme is our result under H₀?

5. **Make your decision**
   - p < α → Reject H₀ (evidence supports H₁)
   - p ≥ α → Fail to reject H₀

**Think of it like a trial:** H₀ = "innocent until proven guilty." The p-value is the strength of the evidence. Alpha is the standard of proof.

### Which Test Do I Use?

| Scenario | Test | Python |
|----------|------|--------|
| Compare one group to a known value | **One-sample t-test** | `stats.ttest_1samp(data, value)` |
| Compare two independent groups | **Two-sample t-test** | `stats.ttest_ind(group1, group2)` |
| Compare same group, before vs. after | **Paired t-test** | `stats.ttest_rel(before, after)` |

---

## 11.9 One-Sample t-Test

**Question:** "Is our sample mean significantly different from a specific value?"

**Use when:** You want to compare one group against a known benchmark or target.

**Examples:**
- "Is our average delivery time really 5 days as advertised?"
- "Is average customer satisfaction above 7.0?"
- "Does our process produce items that weigh 500g on average?" 

In [ ]:
# One-Sample t-Test Example
# A company claims their average call duration is 5 minutes.
# We sampled 50 calls. Let's test the claim.

call_data = pd.read_csv(f"{DATA_URL}/week11/call_durations.csv")
call_times = call_data['Duration']

print("Sample Statistics:")
print(f"  Sample size: {len(call_times)}")
print(f"  Sample mean: {call_times.mean():.2f} minutes")
print(f"  Sample std:  {call_times.std():.2f} minutes")
print(f"  Claimed mean: 5.00 minutes")

In [ ]:
# Perform the one-sample t-test
# H₀: μ = 5.0 (the company's claim is true)
# H₁: μ ≠ 5.0 (actual average differs from claim)

t_stat, p_value = stats.ttest_1samp(call_times, 5.0)

print("One-Sample t-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.4f}")
print(f"  α = 0.05")
print()

if p_value < 0.05:
    print("  Decision: REJECT H₀")
    print("  The average call duration is significantly different from 5 minutes.")
else:
    print("  Decision: FAIL TO REJECT H₀")
    print("  No evidence that call duration differs from 5 minutes.")

In [ ]:
# Visualize the one-sample t-test
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: histogram of the sample data
axes[0].hist(call_times, bins=15, color='steelblue', edgecolor='white',
             alpha=0.7, density=True)
axes[0].axvline(5.0, color='green', linewidth=2.5, linestyle='--',
                label=f'Claimed Mean (5.0)')
axes[0].axvline(call_times.mean(), color='red', linewidth=2.5,
                label=f'Sample Mean ({call_times.mean():.2f})')
axes[0].set_title('Sample Data vs. Claimed Value', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Call Duration (minutes)')
axes[0].legend(fontsize=10)

# Right: where does our test statistic fall on the t-distribution?
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, where=(x >= abs(t_stat)), alpha=0.4, color='coral')
axes[1].fill_between(x, y, where=(x <= -abs(t_stat)), alpha=0.4, color='coral')
axes[1].axvline(t_stat, color='red', linewidth=2, label=f't = {t_stat:.2f}')
axes[1].axvline(-1.96, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(1.96, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title(f'Test Statistic on the Distribution\np-value = {p_value:.4f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-statistic')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## 11.10 Two-Sample t-Test

**Question:** "Are these two independent groups significantly different?"

**Use when:** You're comparing two separate, unrelated groups.

**Examples:**
- "Do customers who saw Ad A spend more than those who saw Ad B?"
- "Is there a salary difference between the IT and Sales departments?"
- "Do North and South region stores have different average sales?" 

In [ ]:
# Two-Sample t-Test Example
# Compare North vs. South region sales

north_sales = pd.read_csv(f"{DATA_URL}/week11/region_north_sales.csv")['Sales']
south_sales = pd.read_csv(f"{DATA_URL}/week11/region_south_sales.csv")['Sales']

print("North Region:")
print(f"  n = {len(north_sales)}, Mean = ${north_sales.mean():,.2f}, Std = ${north_sales.std():,.2f}")
print()
print("South Region:")
print(f"  n = {len(south_sales)}, Mean = ${south_sales.mean():,.2f}, Std = ${south_sales.std():,.2f}")

In [ ]:
# Perform the two-sample t-test
# H₀: μ_North = μ_South (no difference between regions)
# H₁: μ_North ≠ μ_South (regions have different average sales)

t_stat, p_value = stats.ttest_ind(north_sales, south_sales)

print("Two-Sample t-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.4f}")
print(f"  α = 0.05")
print()

if p_value < 0.05:
    print("  Decision: REJECT H₀")
    print("  There IS a significant difference between North and South sales.")
else:
    print("  Decision: FAIL TO REJECT H₀")
    print("  No significant difference detected between regions.")

In [ ]:
# Visualize the two-sample t-test
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: overlapping histograms
axes[0].hist(north_sales, bins=15, alpha=0.6, color='steelblue', edgecolor='white',
             label=f'North (Mean: ${north_sales.mean():,.0f})', density=True)
axes[0].hist(south_sales, bins=15, alpha=0.6, color='coral', edgecolor='white',
             label=f'South (Mean: ${south_sales.mean():,.0f})', density=True)
axes[0].set_title('Sales Distribution by Region', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sales ($)')
axes[0].legend(fontsize=10)

# Right: test statistic on distribution
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, where=(x >= abs(t_stat)), alpha=0.4, color='coral')
axes[1].fill_between(x, y, where=(x <= -abs(t_stat)), alpha=0.4, color='coral')
axes[1].axvline(t_stat, color='red', linewidth=2, label=f't = {t_stat:.2f}')
axes[1].axvline(-1.96, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(1.96, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title(f'Test Statistic\np-value = {p_value:.4f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-statistic')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## 11.11 Paired t-Test

**Question:** "Did the same group change over time? (Before vs. After)"

**Use when:** You're comparing the *same subjects* at two different times or under two conditions.

**Examples:**
- "Did employee performance improve after training?"
- "Did our website redesign increase time-on-page?"
- "Did the new process reduce error rates?"

**Key difference from two-sample:** Here the two measurements are *linked* — same person, same store, same unit.

In [ ]:
# Paired t-Test Example
# A company ran a training program. Did scores improve?

training_data = pd.read_csv(f"{DATA_URL}/week11/training_scores.csv")

print("Training Program Data:")
print(f"  Number of employees: {len(training_data)}")
print(f"  Before training — Mean: {training_data['Before'].mean():.1f}, Std: {training_data['Before'].std():.1f}")
print(f"  After training  — Mean: {training_data['After'].mean():.1f}, Std: {training_data['After'].std():.1f}")
print(f"  Average change:   +{(training_data['After'] - training_data['Before']).mean():.1f} points")

In [ ]:
# Perform the paired t-test
# H₀: μ_diff = 0 (training had no effect)
# H₁: μ_diff ≠ 0 (training changed scores)

t_stat, p_value = stats.ttest_rel(training_data['Before'], training_data['After'])

print("Paired t-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.6f}")
print(f"  α = 0.05")
print()

if p_value < 0.05:
    print("  Decision: REJECT H₀")
    print("  Training significantly changed employee scores!")
else:
    print("  Decision: FAIL TO REJECT H₀")
    print("  No significant effect of training detected.")

In [ ]:
# Visualize the paired t-test
differences = training_data['After'] - training_data['Before']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: before vs after distributions
axes[0].hist(training_data['Before'], bins=12, alpha=0.6, color='steelblue',
             edgecolor='white', label=f'Before (Mean: {training_data["Before"].mean():.1f})',
             density=True)
axes[0].hist(training_data['After'], bins=12, alpha=0.6, color='coral',
             edgecolor='white', label=f'After (Mean: {training_data["After"].mean():.1f})',
             density=True)
axes[0].set_title('Before vs. After', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].legend(fontsize=9)

# Middle: the individual differences
axes[1].hist(differences, bins=12, color='#2ecc71', edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='gray', linestyle='--', linewidth=1.5, label='No Change')
axes[1].axvline(differences.mean(), color='red', linewidth=2.5,
                label=f'Mean Change: +{differences.mean():.1f}')
axes[1].set_title('Individual Differences', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Change in Score')
axes[1].legend(fontsize=9)

# Right: paired lines showing each employee's change
n_show = min(20, len(training_data))
for i in range(n_show):
    color = '#2ecc71' if differences.iloc[i] > 0 else '#e74c3c'
    axes[2].plot([0, 1], [training_data['Before'].iloc[i], training_data['After'].iloc[i]],
                 color=color, alpha=0.5, linewidth=1)
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(['Before', 'After'])
axes[2].set_title(f'Individual Trajectories (first {n_show})',
                  fontsize=12, fontweight='bold')
axes[2].set_ylabel('Score')

plt.suptitle(f'Paired t-Test: t = {t_stat:.2f}, p = {p_value:.6f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 11.12 A/B Testing in Business

**A/B testing is hypothesis testing applied to product decisions.** Every time Netflix tests a new thumbnail, Amazon tests a product page layout, or Google tests a search result format — this is the statistics behind it.

### The A/B Testing Process

1. **Split** users randomly into Group A (control) and Group B (treatment)
2. **Show** each group a different version
3. **Measure** the outcome (clicks, purchases, time on page)
4. **Test** whether the difference is statistically significant
5. **Roll out** the winner (or keep the original if no significant difference)

This is just a two-sample t-test applied to a business decision.

In [ ]:
# A/B Test Simulation: Does a new checkout page design increase spending?
np.random.seed(42)

# Group A: original design
# Group B: new design (with subtle improvements)
group_a = np.random.normal(loc=52, scale=15, size=200)  # original
group_b = np.random.normal(loc=56, scale=15, size=200)  # new design

t_stat, p_value = stats.ttest_ind(group_a, group_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: the two groups
axes[0].hist(group_a, bins=20, alpha=0.6, color='steelblue', edgecolor='white',
             label=f'Original (Mean: ${np.mean(group_a):.2f})', density=True)
axes[0].hist(group_b, bins=20, alpha=0.6, color='coral', edgecolor='white',
             label=f'New Design (Mean: ${np.mean(group_b):.2f})', density=True)
axes[0].set_title('A/B Test: Checkout Page Spending', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Order Value ($)')
axes[0].legend(fontsize=10)

# Right: p-value visualization
x = np.linspace(-4, 4, 1000)
y = norm.pdf(x)
axes[1].plot(x, y, 'k-', linewidth=2)
axes[1].fill_between(x, y, where=(x >= abs(t_stat)), alpha=0.4, color='coral')
axes[1].fill_between(x, y, where=(x <= -abs(t_stat)), alpha=0.4, color='coral')
axes[1].axvline(t_stat, color='red', linewidth=2, label=f't = {t_stat:.2f}')
axes[1].set_title(f'p-value = {p_value:.4f}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-statistic')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"A/B Test Results:")
print(f"  Original design mean: ${np.mean(group_a):.2f}")
print(f"  New design mean:      ${np.mean(group_b):.2f}")
print(f"  Difference:           ${np.mean(group_b) - np.mean(group_a):.2f}")
print(f"  p-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"  → The new design significantly increases spending. Roll it out!")
else:
    print(f"  → No significant difference. Keep the original.")

### Why Sample Size Matters for A/B Tests

A real difference can exist but go **undetected** if your sample is too small. This is called **statistical power** — the ability to detect a real effect when one exists.

In [ ]:
# Power demonstration: same real difference, different sample sizes
np.random.seed(42)
pop_a = np.random.normal(50, 10, 100000)
pop_b = np.random.normal(52, 10, 100000)  # True difference of 2 units

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Same Real Difference — Different Sample Sizes',
             fontsize=14, fontweight='bold')

for ax, n in zip(axes, [10, 50, 500]):
    p_values = []
    for _ in range(1000):
        sa = np.random.choice(pop_a, size=n)
        sb = np.random.choice(pop_b, size=n)
        _, p = stats.ttest_ind(sa, sb)
        p_values.append(p)
    pct_sig = 100 * np.mean(np.array(p_values) < 0.05)
    ax.hist(p_values, bins=30, color='coral', edgecolor='white', alpha=0.7)
    ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
    ax.set_title(f'n = {n} per group\nDetected {pct_sig:.0f}% of the time', fontsize=11)
    ax.set_xlabel('p-value')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("With n=10 per group:  We miss the real difference most of the time.")
print("With n=50 per group:  We catch it sometimes.")
print("With n=500 per group: We detect it nearly every time.")
print("\nLesson: Always plan your sample size BEFORE running an A/B test!")

---

## 11.13 Common Pitfalls and Best Practices

### Mistakes to Avoid

| Mistake | Why It's Wrong |
|---------|---------------|
| "p > 0.05 means there's no effect" | It means we don't have *enough evidence* — absence of evidence ≠ evidence of absence |
| "Small p-value = big effect" | Statistical significance ≠ practical significance. With n=100,000, even a 0.002-minute difference can be "statistically significant" — but it's completely meaningless in practice. "Significant" only means "unlikely to be exactly zero," not "large enough to matter" |
| Choosing α after seeing the data | This is **p-hacking** — set your threshold before you test |
| "Correlation = Causation" | A significant relationship doesn't prove one thing causes another |
| Ignoring sample size | Small samples → wide uncertainty → low power to detect real effects |
| Running many tests without adjusting | If you test 20 hypotheses at α=0.05, each test has a 95% chance of correctly saying "not significant." But the probability that *all 20* are correct is 0.95²⁰ ≈ 0.36, meaning there's a **1 − 0.36 = 64% chance of at least one false positive**. Don't just report the one that "worked" |

### Best Practices

1. **Set α before you look at the data** — no peeking and adjusting
2. **Report effect sizes**, not just p-values — "how big?" matters as much as "is it real?"
3. **Consider practical significance** — a statistically significant $0.50 difference may not matter for business
4. **Use the right test** — one-sample vs. two-sample vs. paired
5. **Check assumptions** — t-tests assume reasonable sample sizes and no extreme outliers
6. **Plan your sample size** — underpowered studies waste time and money

---

## 11.14 Summary

### The Big Ideas

1. **The Central Limit Theorem** is the foundation: sample means follow a normal distribution regardless of the population shape
2. **Larger samples** produce tighter distributions of means (more precise estimates)
3. **More repetitions** produce smoother histograms (higher resolution)
4. **Hypothesis testing** asks: "Is our observed result too unlikely under H₀?"
5. **Alpha** is the threshold we set in advance; **p-value** is what the data gives us
6. **p < α → Reject H₀** ; **p ≥ α → Fail to reject H₀**

### Quick Reference

| Task | Python Code |
|------|-------------|
| One-sample t-test | `stats.ttest_1samp(sample, value)` |
| Two-sample t-test | `stats.ttest_ind(group1, group2)` |
| Paired t-test | `stats.ttest_rel(before, after)` |
| Mean | `df['col'].mean()` |
| Std Dev | `df['col'].std()` |
| Correlation | `df['col1'].corr(df['col2'])` |
| Summary stats | `df.describe()` |

**Import:** `from scipy import stats`

---

## Practice Exercises

Practice applying these concepts with the **Week 11 Practice Assignment**:

1. **Exercise 1:** Descriptive statistics analysis (mean vs. median, skewness)
2. **Exercise 2:** Correlation analysis and interpretation
3. **Exercise 3:** One-sample t-test — test a company's claim
4. **Exercise 4:** Two-sample t-test — compare two regions
5. **Exercise 5:** Paired t-test — evaluate a training program

Then apply everything in the **Integrative Assignment** — a full A/B testing scenario for a fitness app.

---

## Looking Ahead: Week 12

**Linear Regression** — Building Predictive Models

This week you learned to test *if* a difference exists. Next week you'll learn to model *how much* one variable predicts another.

- Simple and multiple linear regression
- Interpreting coefficients (what drives the outcome?)
- Evaluating model fit (R², RMSE)
- Making predictions on new data